# Check Notebook 03b Checkpoint

Load the Notebook 03b output checkpoint and verify all model tensors are finite.

In [1]:

from pathlib import Path
import json
import torch

base = Path('/kaggle/input')
ckpts = sorted(base.rglob('checkpoint_last.pt'))
if not ckpts:
    raise FileNotFoundError('checkpoint_last.pt not found in Kaggle inputs')
ckpt_path = ckpts[0]
print('checkpoint:', ckpt_path)
summary_path = ckpt_path.parent / 'train_summary.json'
config_path = ckpt_path.parent / 'config.json'
summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
config = json.loads(config_path.read_text()) if config_path.exists() else {}
print('summary_core:', json.dumps({k: summary.get(k) for k in ['status','full_epoch_completed','pretrain_completed','posttrain_completed','pretrain_samples','posttrain_samples','test_samples','train_encoded','test_encoded']}, indent=2))
print('config_core:', json.dumps({k: config.get(k) for k in ['learning_rate','posttrain_learning_rate','batch_size','hidden_dim','num_layers','num_heads','action_horizon','state_horizon','num_params']}, indent=2))
ckpt = torch.load(ckpt_path, map_location='cpu')
state = ckpt.get('model_state_dict', {})
num_tensors = 0
num_params = 0
bad = []
absmax = 0.0
for name, tensor in state.items():
    if not torch.is_tensor(tensor):
        continue
    num_tensors += 1
    num_params += tensor.numel()
    if tensor.numel() > 0:
        absmax = max(absmax, float(tensor.detach().abs().max()))
    if not bool(torch.isfinite(tensor).all()):
        bad.append(name)
report = {
    'checkpoint_stage': ckpt.get('stage'),
    'checkpoint_global_step': ckpt.get('global_step'),
    'num_tensors': num_tensors,
    'num_params': num_params,
    'bad_tensor_count': len(bad),
    'bad_tensors_first10': bad[:10],
    'absmax': absmax,
    'weights_all_finite': len(bad) == 0,
}
print('checkpoint_finite_report:', json.dumps(report, indent=2))
Path('/kaggle/working/checkpoint_finite_report.json').write_text(json.dumps({'summary': summary, 'config': config, 'checkpoint_report': report}, indent=2))
if bad:
    raise RuntimeError('Checkpoint contains non-finite tensors')


checkpoint: /kaggle/input/notebooks/kimthanh211005/gr00t-03b-train-dit-h16-4batch-rtx6000/gr00t_official_mini_runs/official_mini_vldit_H16_20subsets_offline_rtx6000/checkpoint_last.pt
summary_core: {
  "status": "completed",
  "full_epoch_completed": true,
  "pretrain_completed": true,
  "posttrain_completed": true,
  "pretrain_samples": 3022069,
  "posttrain_samples": 50000,
  "test_samples": 755702,
  "train_encoded": 3022069,
  "test_encoded": 755702
}
config_core: {
  "learning_rate": 0.0001,
  "posttrain_learning_rate": 5e-05,
  "batch_size": 256,
  "hidden_dim": 512,
  "num_layers": 8,
  "num_heads": 8,
  "action_horizon": 16,
  "state_horizon": 1,
  "num_params": 35272236
}
checkpoint_finite_report: {
  "checkpoint_stage": "posttrain",
  "checkpoint_global_step": 196,
  "num_tensors": 159,
  "num_params": 35272236,
  "bad_tensor_count": 0,
  "bad_tensors_first10": [],
  "absmax": 4.705534934997559,
  "weights_all_finite": true
}
